In [14]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

candidate_paths = [
    Path('marathon-data.csv'),
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Nie znaleziono pliku marathon-data.csv')

df = pd.read_csv(DATA_PATH)
print(DATA_PATH)
df.head()


marathon-data.csv


,age,gender,split,final
0,33,M,01:05:38,02:08:51
1,32,M,01:06:26,02:09:28
2,31,M,01:06:49,02:10:42
3,38,M,01:06:16,02:13:45
4,31,M,01:06:32,02:13:59


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37250 entries, 0 to 37249
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   age     37250 non-null  int64 
 1   gender  37250 non-null  object
 2   split   37250 non-null  object
 3   final   37250 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.1+ MB


In [16]:
df.describe()

,age
count,37250.000000
mean,40.697369
std,10.220043
min,17.000000
25%,33.000000
50%,40.000000
75%,48.000000
max,86.000000


In [17]:
df.describe(include='all').transpose()

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,37250.0,NaN,NaN,NaN,40.697369,10.220043,17.0,33.0,40.0,48.0,86.0
gender,37250,2,M,24665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
split,37250,6496,01:57:55,27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
final,37250,13853,04:36:57,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Wnioski ze statystyk opisowych

Na tym etapie możemy wyciągnąć kilka podstawowych informacji o danych:

- Zbiór zawiera **37250 obserwacji** i nie ma brakujących wartości.
- Jedyną kolumną liczbową jest obecnie `age`, więc klasyczne statystyki dotyczą tylko wieku.

**Wiek zawodników:**
- średni wiek ≈ **40.7 lat**, mediana = **40 lat**,
- 50% zawodników mieści się w przedziale **33–48 lat**,
- zakres wieku: **17–86 lat**,
- rozkład wydaje się dość symetryczny (średnia ≈ mediana).

**Płeć (`gender`):**
- w danych są **2 kategorie**,
- dominującą kategorią jest **M (mężczyźni)**.

**Czasy (`split`, `final`):**
- są zapisane jako **tekst (string)**,
- dlatego na tym etapie nie możemy jeszcze analizować ich jako wartości liczbowych,
- przed dalszą analizą konieczna będzie konwersja do formatu liczbowego (np. minut).

👉 Kluczowy wniosek: zanim zaczniemy analizować wyniki biegu, musimy najpierw **przetworzyć kolumny z czasem na wartości numeryczne**.

In [22]:
for  col in ['split','final']:
    df[col + '_td']= pd.to_timedelta(df[col])
    df[col + '_min'] = df[col + '_td'].dt.total_seconds() / 60

In [19]:
df

,age,gender,split,final,split_td,split_min,final_td,final_min
0,33,M,01:05:38,02:08:51,0 days 01:05:38,65.633333,0 days 02:08:51,128.850000
1,32,M,01:06:26,02:09:28,0 days 01:06:26,66.433333,0 days 02:09:28,129.466667
2,31,M,01:06:49,02:10:42,0 days 01:06:49,66.816667,0 days 02:10:42,130.700000
3,38,M,01:06:16,02:13:45,0 days 01:06:16,66.266667,0 days 02:13:45,133.750000
4,31,M,01:06:32,02:13:59,0 days 01:06:32,66.533333,0 days 02:13:59,133.983333
...,...,...,...,...,...,...,...,...
37245,18,M,04:24:24,09:32:57,0 days 04:24:24,264.400000,0 days 09:32:57,572.950000
37246,36,M,04:35:43,09:33:28,0 days 04:35:43,275.716667,0 days 09:33:28,573.466667
37247,51,M,04:22:35,09:33:40,0 days 04:22:35,262.583333,0 days 09:33:40,573.666667
37248,55,W,04:58:06,10:00:40,0 days 04:58:06,298.100000,0 days 10:00:40,600.666667


In [23]:
df['slowdown_min']=df['final_min'] - 2 * df['split_min']

In [24]:
df

,age,gender,split,final,split_td,split_min,final_td,final_min,slowdown_min
0,33,M,01:05:38,02:08:51,0 days 01:05:38,65.633333,0 days 02:08:51,128.850000,-2.416667
1,32,M,01:06:26,02:09:28,0 days 01:06:26,66.433333,0 days 02:09:28,129.466667,-3.400000
2,31,M,01:06:49,02:10:42,0 days 01:06:49,66.816667,0 days 02:10:42,130.700000,-2.933333
3,38,M,01:06:16,02:13:45,0 days 01:06:16,66.266667,0 days 02:13:45,133.750000,1.216667
4,31,M,01:06:32,02:13:59,0 days 01:06:32,66.533333,0 days 02:13:59,133.983333,0.916667
...,...,...,...,...,...,...,...,...,...
37245,18,M,04:24:24,09:32:57,0 days 04:24:24,264.400000,0 days 09:32:57,572.950000,44.150000
37246,36,M,04:35:43,09:33:28,0 days 04:35:43,275.716667,0 days 09:33:28,573.466667,22.033333
37247,51,M,04:22:35,09:33:40,0 days 04:22:35,262.583333,0 days 09:33:40,573.666667,48.500000
37248,55,W,04:58:06,10:00:40,0 days 04:58:06,298.100000,0 days 10:00:40,600.666667,4.466667


In [25]:
df['slowdown_pct'] = 100 * df['slowdown_min'] / df['final_min']


In [26]:
df

,age,gender,split,final,split_td,split_min,final_td,final_min,slowdown_min,slowdown_pct
0,33,M,01:05:38,02:08:51,0 days 01:05:38,65.633333,0 days 02:08:51,128.850000,-2.416667,-1.875566
1,32,M,01:06:26,02:09:28,0 days 01:06:26,66.433333,0 days 02:09:28,129.466667,-3.400000,-2.626159
2,31,M,01:06:49,02:10:42,0 days 01:06:49,66.816667,0 days 02:10:42,130.700000,-2.933333,-2.244325
3,38,M,01:06:16,02:13:45,0 days 01:06:16,66.266667,0 days 02:13:45,133.750000,1.216667,0.909657
4,31,M,01:06:32,02:13:59,0 days 01:06:32,66.533333,0 days 02:13:59,133.983333,0.916667,0.684165
...,...,...,...,...,...,...,...,...,...,...
37245,18,M,04:24:24,09:32:57,0 days 04:24:24,264.400000,0 days 09:32:57,572.950000,44.150000,7.705733
37246,36,M,04:35:43,09:33:28,0 days 04:35:43,275.716667,0 days 09:33:28,573.466667,22.033333,3.842130
37247,51,M,04:22:35,09:33:40,0 days 04:22:35,262.583333,0 days 09:33:40,573.666667,48.500000,8.454387
37248,55,W,04:58:06,10:00:40,0 days 04:58:06,298.100000,0 days 10:00:40,600.666667,4.466667,0.743618


In [27]:
df['negative_split'] = df['slowdown_min'] < 0


In [28]:
df

,age,gender,split,final,split_td,split_min,final_td,final_min,slowdown_min,slowdown_pct,negative_split
0,33,M,01:05:38,02:08:51,0 days 01:05:38,65.633333,0 days 02:08:51,128.850000,-2.416667,-1.875566,True
1,32,M,01:06:26,02:09:28,0 days 01:06:26,66.433333,0 days 02:09:28,129.466667,-3.400000,-2.626159,True
2,31,M,01:06:49,02:10:42,0 days 01:06:49,66.816667,0 days 02:10:42,130.700000,-2.933333,-2.244325,True
3,38,M,01:06:16,02:13:45,0 days 01:06:16,66.266667,0 days 02:13:45,133.750000,1.216667,0.909657,False
4,31,M,01:06:32,02:13:59,0 days 01:06:32,66.533333,0 days 02:13:59,133.983333,0.916667,0.684165,False
...,...,...,...,...,...,...,...,...,...,...,...
37245,18,M,04:24:24,09:32:57,0 days 04:24:24,264.400000,0 days 09:32:57,572.950000,44.150000,7.705733,False
37246,36,M,04:35:43,09:33:28,0 days 04:35:43,275.716667,0 days 09:33:28,573.466667,22.033333,3.842130,False
37247,51,M,04:22:35,09:33:40,0 days 04:22:35,262.583333,0 days 09:33:40,573.666667,48.500000,8.454387,False
37248,55,W,04:58:06,10:00:40,0 days 04:58:06,298.100000,0 days 10:00:40,600.666667,4.466667,0.743618,False


In [30]:
age_bins = [15, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 100]
age_labels = ['16-19', '20-24', '25-29', '30-34', '35-39', '40-44',
              '45-49', '50-54', '55-59', '60-64', '65-69', '70+']

In [31]:
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels)


In [32]:
df

,age,gender,split,final,split_td,split_min,final_td,final_min,slowdown_min,slowdown_pct,negative_split,age_group
0,33,M,01:05:38,02:08:51,0 days 01:05:38,65.633333,0 days 02:08:51,128.850000,-2.416667,-1.875566,True,30-34
1,32,M,01:06:26,02:09:28,0 days 01:06:26,66.433333,0 days 02:09:28,129.466667,-3.400000,-2.626159,True,30-34
2,31,M,01:06:49,02:10:42,0 days 01:06:49,66.816667,0 days 02:10:42,130.700000,-2.933333,-2.244325,True,30-34
3,38,M,01:06:16,02:13:45,0 days 01:06:16,66.266667,0 days 02:13:45,133.750000,1.216667,0.909657,False,35-39
4,31,M,01:06:32,02:13:59,0 days 01:06:32,66.533333,0 days 02:13:59,133.983333,0.916667,0.684165,False,30-34
...,...,...,...,...,...,...,...,...,...,...,...,...
37245,18,M,04:24:24,09:32:57,0 days 04:24:24,264.400000,0 days 09:32:57,572.950000,44.150000,7.705733,False,16-19
37246,36,M,04:35:43,09:33:28,0 days 04:35:43,275.716667,0 days 09:33:28,573.466667,22.033333,3.842130,False,35-39
37247,51,M,04:22:35,09:33:40,0 days 04:22:35,262.583333,0 days 09:33:40,573.666667,48.500000,8.454387,False,50-54
37248,55,W,04:58:06,10:00:40,0 days 04:58:06,298.100000,0 days 10:00:40,600.666667,4.466667,0.743618,False,55-59


In [33]:
df[['split_min', 'final_min', 'slowdown_min', 'slowdown_pct']].describe().round(2)

,split_min,final_min,slowdown_min,slowdown_pct
count,37250.00,37250.00,37250.00,37250.00
mean,123.91,288.16,40.34,13.09
std,22.92,63.54,23.92,6.18
min,65.35,128.85,-72.47,-23.70
25%,108.42,242.40,22.17,8.83
50%,121.22,284.42,39.35,13.47
75%,136.18,327.60,56.47,17.50
max,299.82,601.13,260.23,57.93
